### **SNA Environment Configuration**
In this block, we configure the environment for Social Network Analysis:

1. **Installing `python-louvain`:** A specific library required to apply the Louvain algorithm (Community Detection), not installed by default.

2. **Importing `networkx`:** The fundamental mathematical engine for building, manipulating, and analyzing graph structures.

3. **Utility Setup:** Importing tools for data management (`pandas`) and process monitoring (`tqdm`).

In [ ]:
!pip install networkx python-louvain

import pandas as pd
import networkx as nx
import community.community_louvain as community_louvain
import networkx.algorithms.community as nx_comm
import os
from tqdm import tqdm

### **Social Network Analysis (SNA)**

In this phase, the goal is to transform the comment dataset into a **social graph** to analyze the structure of user interactions. The analysis focuses on network topology to identify cohesive communities, influential users, and global discussion characteristics.

The workflow rigorously follows network analysis best practices:

1. **Graph Construction:**
* An undirected and weighted graph is generated where **nodes** represent authors and **edges** represent direct replies between users.
* Edge weight increases based on interaction frequency between two users.


2. **Topological Preprocessing:**
* **Self-Loop Removal:** Links from a user to themselves are eliminated to avoid altering centrality metrics.
* **Giant Component Selection:** Analysis is restricted exclusively to the largest connected component. This step is **fundamental** to enable the mathematical calculation of distance-based metrics (such as *Closeness Centrality*, *Diameter*, and *Radius*), which cannot be calculated on a disconnected graph.


3. **Competitive Community Detection:**
To ensure the best possible network partition, two algorithms are compared:
* **Louvain Algorithm:** Heuristic method for modularity optimization.
* **Greedy Modularity (Clauset-Newman-Moore):** Agglomerative hierarchical method.
The code automatically selects the algorithm yielding the highest **Modularity ($Q$)** value.


4. **Metric Calculation:**
Essential metrics are calculated to characterize the network:
* **Centrality:** *Degree*, *Betweenness*, and *Closeness Centrality* for every node.
* **Global Topology:** *Diameter* (maximum width), *Radius* (minimum width), and *Assortativity* (tendency of nodes to connect with similar nodes).

In [ ]:
# ==============================================================================
# PHASE 2: SOCIAL NETWORK ANALYSIS (SNA)
# ==============================================================================

print("--- Social Network Analysis (SNA) ---")

# --- 1. CONFIGURATION ---
BASE_PATH = "/content/drive/MyDrive/MAGISTRALE/Social_Media/SMA_Borgia_Borserini/Datasets"
INPUT_FILE = "chat_control_comments.csv"
OUTPUT_METRICS_ALL = "chat_control_sna.csv"

INPUT_PATH = os.path.join(BASE_PATH, INPUT_FILE)
PATH_ALL = os.path.join(BASE_PATH, OUTPUT_METRICS_ALL)

# Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
except:
    pass

# --- 2. DATA LOADING ---
if not os.path.exists(INPUT_PATH):
    print(f"ERROR: File {INPUT_PATH} does not exist.")
    raise SystemExit

df = pd.read_csv(INPUT_PATH)
df = df.dropna(subset=['comment_id', 'comment_author', 'comment_parent_id'])
print(f"\n[DATA] Loaded {len(df)} valid comments.")

# --- 3. GRAPH CONSTRUCTION ---
print("\n[GRAPH] Building network...")

id_to_author = pd.Series(df.comment_author.values, index=df.comment_id).to_dict()
G = nx.Graph()

for _, row in tqdm(df.iterrows(), total=df.shape[0], desc="Analyzing Interactions"):
    author_source = row['comment_author']
    parent_id_str = str(row['comment_parent_id'])

    if parent_id_str.startswith('t1_'): # Reply to a comment
        parent_id_clean = parent_id_str.split('_')[-1]
        author_target = id_to_author.get(parent_id_clean)

        if author_target:
            # Create or update weighted edge
            if G.has_edge(author_source, author_target):
                G[author_source][author_target]['weight'] += 1
            else:
                G.add_edge(author_source, author_target, weight=1)

# --- GRAPH CLEANING ---
# Remove Self-Loops (users replying to themselves)
self_loops = list(nx.selfloop_edges(G))
G.remove_edges_from(self_loops)
print(f"Removed {len(self_loops)} self-loops.")

# --- GIANT COMPONENT SELECTION ---
# Methodological Requirement: Focus on the largest connected component
# to accurately calculate distance metrics (Closeness, Diameter).
print("\n[FILTER] Selecting Giant Component...")
components = sorted(nx.connected_components(G), key=len, reverse=True)

if len(components) > 0:
    giant_nodes = components[0]
    # Create subgraph containing only the Giant Component
    G_giant = G.subgraph(giant_nodes).copy()

    print(f"Original Nodes: {len(G)}")
    print(f"Giant Component Nodes: {len(G_giant)} ({len(G_giant)/len(G):.1%} of total)")

    # Overwrite G with Giant Component for subsequent analysis
    G = G_giant
else:
    raise SystemExit("The graph is empty!")

# --- 4. COMMUNITY DETECTION COMPARISON ---
print("\n--- COMMUNITY ALGORITHM COMPARISON ---")

# A. Louvain Algorithm (Modularity Optimization)
partition_louvain = community_louvain.best_partition(G, weight='weight')
mod_louvain = community_louvain.modularity(partition_louvain, G, weight='weight')
print(f"1. Louvain -> Modularity: {mod_louvain:.4f} | Communities: {len(set(partition_louvain.values()))}")

# B. Greedy Modularity Algorithm (Clauset-Newman-Moore)
# Note: Uses topology rather than weights directly
greedy_communities = list(nx_comm.greedy_modularity_communities(G))

# Convert list of sets to dictionary format {node: community_id}
partition_greedy = {}
for i, comm in enumerate(greedy_communities):
    for node in comm:
        partition_greedy[node] = i

# Calculate modularity for Greedy
mod_greedy = community_louvain.modularity(partition_greedy, G, weight='weight')
print(f"2. Greedy Modularity -> Modularity: {mod_greedy:.4f} | Communities: {len(greedy_communities)}")

# SELECTION: Choose the algorithm with the highest modularity
if mod_louvain >= mod_greedy:
    print(">> WINNER: Louvain Algorithm")
    best_partition = partition_louvain
else:
    print(">> WINNER: Greedy Modularity")
    best_partition = partition_greedy

# --- 5. CENTRALITY METRICS CALCULATION ---
print("\n[METRICS] Calculating centrality and distances...")

# A. Degree Centrality
degree_cent = nx.degree_centrality(G)

# B. Closeness Centrality
print("   - Calculating Closeness...")
closeness_cent = nx.closeness_centrality(G)

# C. Betweenness Centrality
print("   - Calculating Betweenness...")
between_cent = nx.betweenness_centrality(G, weight='weight')

# --- PRINT TOP INFLUENCERS ---
def print_top_nodes(metric_dict, metric_name, n=5):
    sorted_nodes = sorted(metric_dict.items(), key=lambda x: x[1], reverse=True)[:n]
    print(f"\n   [TOP {n} {metric_name}]")
    for rank, (node, val) in enumerate(sorted_nodes, 1):
        print(f"   {rank}. {node} ({val:.4f})")

print_top_nodes(degree_cent, "Degree Centrality (Hubs)")
print_top_nodes(between_cent, "Betweenness Centrality (Bridges)")
print_top_nodes(closeness_cent, "Closeness Centrality (Speed)")

# --- 6. GLOBAL METRICS (Diameter & Radius) ---
print("   - Calculating Diameter and Radius...")
try:
    # Eccentricity is required for diameter and radius
    eccentricity = nx.eccentricity(G)
    diameter = nx.diameter(G, e=eccentricity)
    radius = nx.radius(G, e=eccentricity)
    print(f">> Network Diameter: {diameter}")
    print(f">> Network Radius: {radius}")
except Exception as e:
    print(f">> Unable to calculate Diameter/Radius (Graph too large or disconnected): {e}")

# --- 7. ASSORTATIVITY ---
assortativity = nx.degree_assortativity_coefficient(G)
print(f"Assortativity Coefficient: {assortativity:.4f}")

# --- 8. FINAL DATASET EXPORT ---
print("\n[EXPORT] Saving data...")

# Create DataFrame with metrics
df_metrics = pd.DataFrame({
    'comment_author': list(best_partition.keys()),
    'community': list(best_partition.values()),
    'degree_centrality': [degree_cent[n] for n in best_partition.keys()],
    'closeness_centrality': [closeness_cent[n] for n in best_partition.keys()],
    'betweenness_centrality': [between_cent[n] for n in best_partition.keys()]
})

# Merge with original dataframe
# Use inner join to filter out nodes not in the Giant Component
df_enriched_all = df[df['comment_author'].isin(list(G.nodes()))].merge(
    df_metrics,
    on='comment_author',
    how='left'
)

# Calculate Community Size
comm_size_map = df_enriched_all.groupby('community')['comment_author'].nunique()
df_enriched_all['community_users_count'] = df_enriched_all['community'].map(comm_size_map)

# Save to CSV
df_enriched_all.to_csv(PATH_ALL, index=False)
print(f"File saved successfully: {PATH_ALL}")

--- Social Network Analysis (SNA) ---
Mounted at /content/drive

[DATA] Loaded 9031 valid comments.

[GRAPH] Building network...


Analyzing Interactions: 100%|██████████| 9031/9031 [00:00<00:00, 16867.39it/s]


Removed 7 self-loops.

[FILTER] Selecting Giant Component...
Original Nodes: 3182
Giant Component Nodes: 2940 (92.4% of total)

--- COMMUNITY ALGORITHM COMPARISON ---
1. Louvain -> Modularity: 0.7980 | Communities: 38
2. Greedy Modularity -> Modularity: 0.7483 | Communities: 35
>> WINNER: Louvain Algorithm

[METRICS] Calculating centrality and distances...
   - Calculating Closeness...
   - Calculating Betweenness...

   [TOP 5 Degree Centrality (Hubs)]
   1. Dry_Row_7050 (0.0316)
   2. EmbarrassedHelp (0.0282)
   3. silentspectator27 (0.0231)
   4. KN_Knoxxius (0.0228)
   5. SeriouslyNotSerious2 (0.0184)

   [TOP 5 Betweenness Centrality (Bridges)]
   1. EmbarrassedHelp (0.1850)
   2. Dry_Row_7050 (0.1577)
   3. Nattekat (0.0854)
   4. KN_Knoxxius (0.0673)
   5. silentspectator27 (0.0650)

   [TOP 5 Closeness Centrality (Speed)]
   1. EmbarrassedHelp (0.2793)
   2. Dry_Row_7050 (0.2613)
   3. silentspectator27 (0.2574)
   4. Frosty-Cell (0.2565)
   5. michael0n (0.2545)
   - Calculati

### **Results SNA**

### 1. Network Coverage (Giant Component: 92.4%)

* **Data:** 2,940 out of 3,182 nodes are within the giant component.
* **Interpretation:** The discussion is **strongly unified**. There are no significant isolated groups talking amongst themselves while ignoring the rest of the network. Almost everyone (92.4%) is part of the same macro-debate. This methodologically legitimizes the choice to analyze only the Giant Component, as the data loss (the excluded nodes) is negligible.

### 2. Community Structure (Modularity: 0.7980)

* **Data:** Modularity is **0.7980** with **38 distinct communities**.
* **Interpretation:** This value is **extremely high**. Typically, values above 0.3 indicate a good community structure. A value of ~0.8 indicates that the network is **sharply fragmented into closed groups**.
* **Sociological Meaning:** Users tend to interact almost exclusively within their own group. This is a strong indicator of **Echo Chambers** or distinct factions that do not engage in much "cross-talk."
* **Comparison:** The **Louvain** algorithm outperformed Greedy Modularity (0.7980 vs 0.7483) and identified 38 communities, optimizing the partition better than topology-based methods.

### 3. Network Shape (Diameter: 17, Radius: 9)

* **Data:** It takes 17 steps to travel between the two most distant nodes (A to B).
* **Interpretation:** The network is "elongated." In a "Small World" network (like Facebook friendships), the diameter is often around 6. A diameter of 17 suggests that discussions are **deep** (long back-and-forth reply chains) rather than broad and flat. Information travels slowly from one end of the network to the other.

### 4. Hierarchy (Assortativity: -0.1504)

* **Data:** Negative value (-0.15).
* **Interpretation:** The network is **Disassortative**.
* **Meaning:** "Famous" nodes (Hubs with many connections) do not connect with other famous nodes; instead, they tend to connect with many "small" nodes (common users).
* **Model:** This represents a **"Core-Periphery"** or "Hub and Spoke" structure. Imagine an influencer (e.g., *Dry_Row_7050* or *EmbarrassedHelp*) at the center, surrounded by a crowd of users replying to them but not to each other. It is not an "elite club" where hubs only talk to hubs.

### 5. Key Actors Analysis (Top Influencers)

* **Degree Centrality:** A minimal number of hubs (led by *Dry_Row_7050*) concentrate the vast majority of connections, sustaining the conversation while the average user participates marginally.

* **Betweenness Centrality:** The top score of **0.1850** (held by *EmbarrassedHelp*) reveals the presence of critical **"gatekeepers"**. These actors are structurally essential to bridge the 38 distinct communities, preventing the network from fragmenting into disconnected islands.

* **Closeness Centrality:** High scores indicate that, despite the network's large diameter (17), the core influencers are positioned to disseminate information rapidly to the entire Giant Component.